# prime-rl S30: minimal SYS prompt (S22 style)

S29 観察:
- SYS で「PROCEDURE (mandatory): 1. FIRST call compute_index_delta...」と procedural に強調したのに
  model は **lookup tool を一切呼ばず** terminal 直行
- inference.log / orchestrator.log で lookup tool 呼びゼロ
- → 過剰な procedural 指示が逆効果、450M モデルは長い指示文で混乱して terminal 直行モードになっている疑い

S22 era (model が実際に tool 呼んでた時代) の SYS は遥かに簡素:
- Goals: bandwidth / image attach / terminate exactly one
- Style: think briefly, one tool call per step, stop when enough info
- 具体的 tool 名すら書いてない、PROCEDURE なし、MUST なし

S30: **SYS を S22 style に戻す**。
- env / reward は S29 のまま (lookup_called +0.2、reason_grounded_correct +0.5、balanced 1.0/3.0)
- env push 不要 (notebook 編集のみ)

注目指標:
- **lookup tool 呼び数** (orchestrator.log) — S29 ゼロから増えるか
- accuracy / positive negative balance
- frac_grounded (lookup 呼んだ後 reason に index 名出るか)
- base ≠ after_grpo (variance 信号で GRPO 動くか)


## 1. Install + 2 patches + DATA_ROOT (S6/S9/S10 と同じ recipe)

In [ ]:
import os, subprocess, glob, re

PREP_MAIN = "/kaggle/input/notebooks/titanic12/prime-rl-offline-prep"
PREP_ENV  = "/kaggle/input/notebooks/titanic12/satelliteagent-env-prep"
BASE_MAIN = f"{PREP_MAIN}/output" if os.path.exists(f"{PREP_MAIN}/output") else PREP_MAIN
BASE_ENV  = f"{PREP_ENV}/output"  if os.path.exists(f"{PREP_ENV}/output")  else PREP_ENV
WHEELS_MAIN = f"{BASE_MAIN}/wheels"
WHEELS_ENV  = f"{BASE_ENV}/wheels"
MODEL_DIR = f"{BASE_MAIN}/models/LFM2.5-VL-450M"
VISION_ATTR = "model.vision_tower"
LM_ATTR     = "model.language_model"

SKIP_PREFIXES = (
    "torch-", "torchvision-", "torchaudio-", "nvidia_",
    "numpy-", "scipy-", "scikit_learn-", "pandas-",
)

def parse_version(name):
    m = re.match(r"([a-zA-Z0-9_]+?)-(\d+(?:\.\d+)*)", os.path.basename(name))
    if not m: return None, None
    return m.group(1).lower(), tuple(int(x) for x in m.group(2).split("."))

all_wheels = sorted(glob.glob(f"{WHEELS_MAIN}/*.whl") + glob.glob(f"{WHEELS_ENV}/*.whl"))
remaining = [w for w in all_wheels if not os.path.basename(w).startswith(SKIP_PREFIXES)]
keep = {}
for w in remaining:
    pkg, ver = parse_version(w)
    if pkg is None:
        keep[os.path.basename(w)] = (None, w); continue
    if pkg in keep and keep[pkg][0] is not None:
        if ver > keep[pkg][0]:
            keep[pkg] = (ver, w)
    else:
        keep[pkg] = (ver, w)
filtered = [v[1] for v in keep.values()]
print(f"installing {len(filtered)} wheels")
subprocess.run(
    "python3.12 -m pip install --no-index --no-build-isolation --pre --no-deps "
    + " ".join(filtered),
    shell=True, check=True,
)

patch_registry = '''
import inspect
import prime_rl.utils.vlm as v
path = inspect.getfile(v)
content = open(path).read()
if "lfm2_vl" not in content:
    needle = "    \\"qwen3_vl\\":"
    insertion = (
        "    \\"lfm2_vl\\": VLMModelInfo(vision_encoder_attr=\\"model.vision_tower\\", language_model_attr=\\"model.language_model\\"),\\n"
        "    \\"qwen3_vl\\":"
    )
    open(path, "w").write(content.replace(needle, insertion, 1))
    print(f"PATCH 1 ok: {path}")
else:
    print("PATCH 1 skip")
'''
subprocess.run(["python3.12", "-c", patch_registry], check=True)

patch_moe_guard = '''
import inspect
import prime_rl.trainer.model as m
path = inspect.getfile(m)
content = open(path).read()
old_block = "    for transformer_block in language_model.layers:\\n        if not isinstance(transformer_block.mlp, (MoE, LatentMoE)):\\n            continue\\n        transformer_block.mlp.set_ep_comm_backend(backend)\\n        transformer_block.mlp.set_deepep_token_chunk_size(config.deepep_token_chunk_size)"
new_block = "    for transformer_block in language_model.layers:\\n        mlp = getattr(transformer_block, \\"mlp\\", None)\\n        if mlp is None or not isinstance(mlp, (MoE, LatentMoE)):\\n            continue\\n        mlp.set_ep_comm_backend(backend)\\n        mlp.set_deepep_token_chunk_size(config.deepep_token_chunk_size)"
if old_block in content:
    open(path, "w").write(content.replace(old_block, new_block))
    print(f"PATCH 2 ok: {path}")
elif "mlp = getattr(transformer_block" in content:
    print("PATCH 2 skip (already)")
else:
    print("PATCH 2 WARN: needle not found")
'''
subprocess.run(["python3.12", "-c", patch_moe_guard], check=True)

subprocess.run(
    "python3.12 -c 'from prime_rl.utils.vlm import VLM_REGISTRY; print(\"VLM_REGISTRY:\", list(VLM_REGISTRY.keys()))'",
    shell=True, check=True,
)

# --- DATA_ROOT (raw-v2: canonical_dataset.yaml + curated_pairs/) ---
DATA_CANDIDATES = [
    "/kaggle/input/satelliteagent-raw-v2",
    "/kaggle/input/datasets/titanic12/satelliteagent-raw-v2",
]
DATA_ROOT = None
for cand in DATA_CANDIDATES:
    if os.path.isfile(f"{cand}/canonical_dataset.yaml"):
        DATA_ROOT = cand; break
if DATA_ROOT is None:
    found = subprocess.check_output(
        "find /kaggle/input -name canonical_dataset.yaml 2>/dev/null | head -1",
        shell=True,
    ).decode().strip()
    DATA_ROOT = os.path.dirname(found)
print(f"DATA_ROOT       = {DATA_ROOT}")

# --- PRECOMPUTE_ROOT (precompute-v2: per-case tool response cache) ---
PC_CANDIDATES = [
    "/kaggle/input/satelliteagent-precompute-v2",
    "/kaggle/input/datasets/titanic12/satelliteagent-precompute-v2",
]
PRECOMPUTE_ROOT = None
for cand in PC_CANDIDATES:
    if os.path.isdir(cand):
        nfound = subprocess.check_output(
            f"find {cand} -name 'classify_change.yaml' 2>/dev/null | wc -l",
            shell=True,
        ).decode().strip()
        if int(nfound or "0") > 0:
            PRECOMPUTE_ROOT = cand; break
if PRECOMPUTE_ROOT is None:
    found = subprocess.check_output(
        "find /kaggle/input -name 'classify_change.yaml' 2>/dev/null | head -1",
        shell=True,
    ).decode().strip()
    if found:
        # path = .../<case_id>/classify_change.yaml -> root = parent of <case_id>
        PRECOMPUTE_ROOT = os.path.dirname(os.path.dirname(found))
if PRECOMPUTE_ROOT is None:
    raise RuntimeError("satelliteagent-precompute-v2 not found under /kaggle/input/")
print(f"PRECOMPUTE_ROOT = {PRECOMPUTE_ROOT}")
sample_case_dir = subprocess.check_output(
    f"ls {PRECOMPUTE_ROOT} 2>/dev/null | head -1", shell=True,
).decode().strip()
print(f"  sample case dir: {sample_case_dir}")

MAX_LEN = 8192

# S30 system prompt: minimal S22-style.
# S26-29 で procedural 指示を盛りすぎて model が tool 呼ばなくなった反省。
# S22 era の動いてた版に戻す。reason 引数の note のみ追加 (env が要求)。
SYS = (
    "You are an onboard satellite operator agent running on NVIDIA Orin 16GB.\n\n"
    "You are shown a Sentinel-2 image pair (before = previous pass, after = current pass) of the same\n"
    "location. Your job is to decide what to report to the ground station.\n\n"
    "Goals:\n"
    "- Minimize downlink bandwidth usage while preserving critical information.\n"
    "- Attach the raw image only when text alone cannot convey the situation.\n"
    "- Always terminate with exactly one of: submit_to_ground(...) or drop().\n\n"
    "Both submit_to_ground and drop take a required `reason` argument.\n\n"
    "Style:\n"
    "- Think briefly in natural language before each tool call (one sentence).\n"
    "- One tool call per step.\n"
    "- Stop as soon as you have enough information.\n"
)


## 2. Eval helper (S10 v5 と同じ、process-group kill 付き)

In [ ]:
import os, json, time, base64, signal, subprocess, urllib.request, urllib.error, re

EVAL_RESULTS = []

# S28: reason still required, but tool descriptions no longer include
# concrete examples (model copies them verbatim).
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "submit_to_ground",
            "description": "Transmit a satellite report to the ground station. Use when the scene is urgent / important / requires human attention. `reason` must cite the spectral evidence you actually observed via your lookup tools.",
            "parameters": {
                "type": "object",
                "properties": {
                    "report_id":    {"type": "string"},
                    "reason":       {"type": "string", "description": "Justification citing observed spectral evidence (index name + numerical change you saw)."},
                    "attach_image": {"type": "boolean"},
                    "urgency":      {"type": "string", "enum": ["low", "medium", "high"]},
                    "change_type":  {"type": "string"},
                },
                "required": ["report_id", "reason"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "drop",
            "description": "Drop the data without transmitting. Use when the scene is uninteresting and not worth bandwidth. `reason` must cite the spectral evidence supporting the no-change decision.",
            "parameters": {
                "type": "object",
                "properties": {
                    "reason": {"type": "string", "description": "Justification citing observed spectral evidence."},
                },
                "required": ["reason"],
            },
        },
    },
]

def _b64_image(path):
    with open(path, "rb") as f:
        return "data:image/png;base64," + base64.b64encode(f.read()).decode("ascii")

def _build_messages(case):
    case_id = case["id"]
    before = f"{DATA_ROOT}/curated_pairs/{case_id}/before.png"
    after  = f"{DATA_ROOT}/curated_pairs/{case_id}/after.png"
    return [
        {"role": "system", "content": [{"type": "text", "text": SYS}]},
        {"role": "user", "content": [
            {"type": "text",      "text": "Before image (previous satellite pass over this location):"},
            {"type": "image_url", "image_url": {"url": _b64_image(before)}},
            {"type": "text",      "text": "After image (current pass, same location):"},
            {"type": "image_url", "image_url": {"url": _b64_image(after)}},
            {"type": "text",      "text": (
                "Analyze the pair and decide what to report to ground. "
                "Use your tools. End with submit_to_ground(...) or drop()."
            )},
        ]},
    ]

def _expected_action(case):
    return "submit_to_ground" if case.get("type") == "positive" else "drop"

def _parse_tool_name(message):
    tcs = message.get("tool_calls") or []
    if tcs:
        return tcs[0].get("function", {}).get("name")
    content = message.get("content") or ""
    if isinstance(content, list):
        content = "".join(p.get("text", "") for p in content if isinstance(p, dict))
    m = re.search(r'"name"\s*:\s*"([^"]+)"', content)
    return m.group(1) if m else None

def _parse_tool_reason(message):
    tcs = message.get("tool_calls") or []
    if not tcs:
        return None
    args = (tcs[0].get("function") or {}).get("arguments")
    if isinstance(args, str):
        try: args = json.loads(args)
        except Exception: return None
    if isinstance(args, dict):
        return args.get("reason")
    return None

def _gpu_memory_used_mb():
    try:
        out = subprocess.check_output(
            "nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
            shell=True,
        ).decode().strip().splitlines()[0]
        return int(out)
    except Exception:
        return -1

def _wait_gpu_free(threshold_mb=8000, max_wait_s=180):
    deadline = time.time() + max_wait_s
    last = _gpu_memory_used_mb()
    while time.time() < deadline:
        used = _gpu_memory_used_mb()
        last = used
        if 0 <= used < threshold_mb:
            return used
        time.sleep(2)
    return last

def _hard_kill_vllm(p):
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
    except (ProcessLookupError, OSError): pass
    try: p.wait(timeout=5)
    except subprocess.TimeoutExpired: pass
    if p.poll() is None:
        try: os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except (ProcessLookupError, OSError): pass
        try: p.wait(timeout=5)
        except subprocess.TimeoutExpired: pass
    for pat in ["prime_rl.entrypoints.inference", "vllm.entrypoints", "vllm", "EngineCore", "MQLLMEngine"]:
        subprocess.run(f"pkill -9 -f '{pat}' 2>/dev/null || true", shell=True)

def run_eval(model_path, label, *, port=8001, n_cases=None, eval_temperature=0.0):
    print(f"\n=== EVAL [{label}] starting ===")
    print(f"model_path = {model_path}")
    print(f"GPU mem before = {_gpu_memory_used_mb()} MB")
    waited = _wait_gpu_free(8000, 180)
    print(f"GPU mem after wait = {waited} MB")

    eval_infer = f'''gpu_memory_utilization = 0.45

[model]
name = "{model_path}"
max_model_len = {MAX_LEN}
enforce_eager = true
tool_call_parser = "hermes"

[server]
port = {port}
'''
    toml_path = f"/kaggle/working/outputs/eval_{label}.toml"
    log_path  = f"/kaggle/working/outputs/proc_logs/eval_{label}.log"
    os.makedirs("/kaggle/working/outputs/proc_logs", exist_ok=True)
    with open(toml_path, "w") as f: f.write(eval_infer)

    env = os.environ.copy()
    env.update({
        "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
        "WANDB_MODE": "disabled", "VLLM_API_KEY": "dummy",
        "CUDA_VISIBLE_DEVICES": "0",
    })

    lf = open(log_path, "wb")
    p = subprocess.Popen(
        f"python3.12 -m prime_rl.entrypoints.inference @ {toml_path}",
        shell=True, env=env, stdout=lf, stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    try:
        ready = False; deadline = time.time() + 600
        while time.time() < deadline:
            if p.poll() is not None:
                raise RuntimeError(f"vLLM died early. Tail:\n{subprocess.check_output(f'tail -n 60 {log_path}', shell=True).decode(errors='replace')}")
            try:
                req = urllib.request.Request(f"http://localhost:{port}/v1/models", headers={"Authorization": "Bearer dummy"})
                if urllib.request.urlopen(req, timeout=5).status == 200:
                    ready = True; break
            except Exception: pass
            time.sleep(5)
        if not ready:
            raise RuntimeError(f"vLLM did not become ready for {label}")
        print(f"[{label}] vLLM up, GPU mem = {_gpu_memory_used_mb()} MB")

        try:
            req = urllib.request.Request(f"http://localhost:{port}/v1/models", headers={"Authorization": "Bearer dummy"})
            with urllib.request.urlopen(req, timeout=10) as r:
                served_model_name = json.loads(r.read())["data"][0]["id"]
        except Exception:
            served_model_name = model_path
        print(f"[{label}] served model id = {served_model_name}, eval_temperature = {eval_temperature}")

        import yaml
        canonical = yaml.safe_load(open(f"{DATA_ROOT}/canonical_dataset.yaml"))
        all_cases = canonical["cases"]
        cases = all_cases[:n_cases] if n_cases else all_cases

        per_case_correct = {"positive": [0, 0], "negative": [0, 0]}
        confusion = {}
        samples = []
        n_with_index_term = 0
        n_with_reason = 0
        index_terms = ("nbr", "ndvi", "ndwi", "mndwi", "ndbi", "ndsi", "delta", "frac_decrease", "frac_increase", "swir", "nir", "rededge")
        t0 = time.time()
        for case in cases:
            body = {
                "model": served_model_name,
                "messages": _build_messages(case),
                "tools": TOOLS,
                "tool_choice": "required",
                "max_tokens": 256,
                "temperature": eval_temperature,
            }
            req = urllib.request.Request(
                f"http://localhost:{port}/v1/chat/completions",
                data=json.dumps(body).encode("utf-8"),
                headers={"Content-Type": "application/json", "Authorization": "Bearer dummy"},
            )
            try:
                with urllib.request.urlopen(req, timeout=120) as r:
                    resp = json.loads(r.read())
                msg = resp["choices"][0]["message"]
                got = _parse_tool_name(msg)
                got_reason = _parse_tool_reason(msg)
            except urllib.error.HTTPError as e:
                got = f"HTTP_{e.code}"
                got_reason = None
                msg = {"error": e.read().decode(errors='replace')[:200]}

            expected = _expected_action(case)
            is_correct = (got == expected)
            kind = case.get("type") or "positive"
            per_case_correct[kind][1] += 1
            if is_correct: per_case_correct[kind][0] += 1
            confusion[(expected, got)] = confusion.get((expected, got), 0) + 1
            if isinstance(got_reason, str) and len(got_reason.strip()) >= 8:
                n_with_reason += 1
                if any(t in got_reason.lower() for t in index_terms):
                    n_with_index_term += 1
            if len(samples) < 3 or (not is_correct and len(samples) < 6):
                samples.append({
                    "id": case["id"], "expected": expected, "got": got, "ok": is_correct,
                    "reason": (got_reason[:200] if isinstance(got_reason, str) else got_reason),
                })

        elapsed = time.time() - t0
        total_correct = per_case_correct["positive"][0] + per_case_correct["negative"][0]
        total_n = per_case_correct["positive"][1] + per_case_correct["negative"][1]
        result = {
            "label": label, "model_path": model_path, "eval_temperature": eval_temperature,
            "n_total": total_n, "n_correct": total_correct,
            "accuracy": total_correct / total_n if total_n else 0.0,
            "n_with_reason": n_with_reason,
            "n_with_index_term": n_with_index_term,
            "frac_grounded": (n_with_index_term / total_n) if total_n else 0.0,
            "positive": {
                "n": per_case_correct["positive"][1],
                "correct": per_case_correct["positive"][0],
                "acc": (per_case_correct["positive"][0] / per_case_correct["positive"][1]) if per_case_correct["positive"][1] else 0.0,
            },
            "negative": {
                "n": per_case_correct["negative"][1],
                "correct": per_case_correct["negative"][0],
                "acc": (per_case_correct["negative"][0] / per_case_correct["negative"][1]) if per_case_correct["negative"][1] else 0.0,
            },
            "confusion": {f"{k[0]}->{k[1]}": v for k, v in sorted(confusion.items(), key=lambda x: -x[1])},
            "samples": samples,
            "elapsed_s": round(elapsed, 1),
        }
        EVAL_RESULTS.append(result)
        print(json.dumps(result, indent=2, ensure_ascii=False))
        return result
    finally:
        _hard_kill_vllm(p)
        lf.close()
        freed = _wait_gpu_free(8000, 180)
        print(f"[{label}] GPU mem after vLLM stop = {freed} MB")


## 3. GRPO from base (max_steps=100, rollouts=8, temperature=0.7, max_turns=5)

trainer.toml に `[ckpt]` を入れて `outputs/weights/step_100/` に safetensors を吐く
(after_grpo eval の base にする)。orch.toml の `[train.sampling] temperature=0.7`
で rollout 多様性を確保。S18 は **max_turns を 5 に拡張**してモデルが lookup→terminal
の余裕を持てるようにする。


In [ ]:
import os, subprocess

GRPO_STEPS = 200
ROLLOUTS_PER_EXAMPLE = 8
TRAIN_TEMPERATURE = 1.0
BATCH_SIZE = ROLLOUTS_PER_EXAMPLE
ENV_MAX_TURNS = 5

TRAINER_TOML = f'''max_steps = {GRPO_STEPS}

[model]
name = "{MODEL_DIR}"
seq_len = {MAX_LEN}
attn = "sdpa"
optimization_dtype = "bfloat16"
reduce_dtype = "bfloat16"

[model.vlm]
vision_encoder_attr = "{VISION_ATTR}"
language_model_attr = "{LM_ATTR}"

[optim]
lr = 1e-6

[ckpt]
interval = {GRPO_STEPS}

[ckpt.weights]
save_format = "safetensors"
save_sharded = false
'''

INFER_TOML = f'''gpu_memory_utilization = 0.5

[model]
name = "{MODEL_DIR}"
max_model_len = {MAX_LEN}
enforce_eager = true
tool_call_parser = "hermes"

[server]
port = 8000
'''

# orch.toml: pass SYS into env via [train.env.args] system_prompt.
# TOML literal multi-line string ('''...''') preserves SYS verbatim
# (no escape processing). Will fail loudly if SYS contains "'''".
assert "'''" not in SYS, "SYS contains ''' which would break TOML literal string"

ORCH_TOML = f'''max_steps = {GRPO_STEPS}
batch_size = {BATCH_SIZE}
seq_len = {MAX_LEN}
rollouts_per_example = {ROLLOUTS_PER_EXAMPLE}
filters = []

[model]
name = "{MODEL_DIR}"

[train.sampling]
max_completion_tokens = 256
temperature = {TRAIN_TEMPERATURE}

[train.sampling.extra_body]
tool_choice = "required"

[[train.env]]
id = "satelliteagent_env"

[train.env.args]
toy = false
data_root = "{DATA_ROOT}"
precompute_root = "{PRECOMPUTE_ROOT}"
max_turns = {ENV_MAX_TURNS}
system_prompt = """{SYS}"""
'''

os.makedirs("/kaggle/working/outputs", exist_ok=True)
trainer_toml = "/kaggle/working/outputs/trainer.toml"
infer_toml   = "/kaggle/working/outputs/infer.toml"
orch_toml    = "/kaggle/working/outputs/orch.toml"
with open(trainer_toml, "w") as f: f.write(TRAINER_TOML)
with open(infer_toml,   "w") as f: f.write(INFER_TOML)
with open(orch_toml,    "w") as f: f.write(ORCH_TOML)
for label, body in [("trainer.toml", TRAINER_TOML), ("infer.toml", INFER_TOML), ("orch.toml", ORCH_TOML)]:
    print(f"=== {label} ===")
    print(body)

def _validate(label, module, toml_path):
    print(f"  [{label}] validating {toml_path}...")
    r = subprocess.run(
        f"python3.12 -m {module} @ {toml_path} --help",
        shell=True, capture_output=True, text=True,
    )
    out = (r.stdout or "") + (r.stderr or "")
    bad_markers = ["validation error", "Failed to validate", "Config file error"]
    if any(m in out for m in bad_markers) and "Usage:" not in out:
        print(out[-2000:])
        raise RuntimeError(f"{label} config validation failed; see message above")
    if r.returncode != 0 and "Usage:" not in out:
        print(out[-2000:])
        raise RuntimeError(f"{label} CLI exited {r.returncode} during validation")
    print(f"  [{label}] OK")

print("\n=== Validating TOMLs ===")
_validate("orchestrator", "prime_rl.orchestrator.orchestrator", orch_toml)
_validate("trainer",      "prime_rl.trainer.rl.train",          trainer_toml)
print("All TOMLs validated.")


In [ ]:
import os, subprocess, time, urllib.request

LOG_DIR = "/kaggle/working/outputs/proc_logs"
os.makedirs(LOG_DIR, exist_ok=True)
API_KEY = "dummy"

PREP_MAIN = "/kaggle/input/notebooks/titanic12/prime-rl-offline-prep"
BASE_MAIN = f"{PREP_MAIN}/output" if os.path.exists(f"{PREP_MAIN}/output") else PREP_MAIN
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)

env = os.environ.copy()
env.update({
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1",
    "WANDB_MODE": "disabled", "CUDA_VISIBLE_DEVICES": "0", "VLLM_API_KEY": API_KEY,
    "HF_HOME": f"{BASE_MAIN}/hf_cache", "HF_HUB_CACHE": f"{BASE_MAIN}/hf_cache",
    "HF_DATASETS_CACHE": "/kaggle/working/hf_cache",
    # per-rollout chat trace + tool_call_log dumped as JSONL
    "SATELLITEAGENT_TRACE_PATH": "/kaggle/working/outputs/satelliteagent_traces.jsonl",
})

procs = {}
def start_bg(name, cmd):
    log_path = f"{LOG_DIR}/{name}.log"
    log_f = open(log_path, "wb")
    p = subprocess.Popen(cmd, shell=True, env=env, stdout=log_f, stderr=subprocess.STDOUT,
                         start_new_session=True)
    procs[name] = (p, log_f, log_path)
    print(f"[{name}] started pid={p.pid} log={log_path}")
    return p

def cleanup():
    import signal as _s
    for name, (p, f, _) in procs.items():
        if p.poll() is None:
            try: os.killpg(os.getpgid(p.pid), _s.SIGTERM)
            except: pass
            try: p.wait(timeout=10)
            except: pass
            if p.poll() is None:
                try: os.killpg(os.getpgid(p.pid), _s.SIGKILL)
                except: pass
        f.close()
    for pat in ["prime_rl.entrypoints.inference", "prime_rl.orchestrator", "prime_rl.trainer", "vllm.entrypoints", "EngineCore"]:
        subprocess.run(f"pkill -9 -f '{pat}' 2>/dev/null || true", shell=True)

def tail(path, n=80):
    try: return subprocess.check_output(f"tail -n {n} {path}", shell=True).decode(errors="replace")
    except Exception as e: return f"<tail error: {e}>"

outcome = "FAIL"
try:
    start_bg("inference", "python3.12 -m prime_rl.entrypoints.inference @ /kaggle/working/outputs/infer.toml")
    print("\nWaiting for vLLM ...")
    ready = False; deadline = time.time() + 600
    while time.time() < deadline:
        if procs["inference"][0].poll() is not None:
            raise RuntimeError(f"inference died early. Tail:\n{tail(procs['inference'][2], 80)}")
        try:
            req = urllib.request.Request("http://localhost:8000/v1/models", headers={"Authorization": f"Bearer {API_KEY}"})
            if urllib.request.urlopen(req, timeout=5).status == 200:
                ready = True; break
        except: pass
        time.sleep(5)
    if not ready:
        raise RuntimeError(f"inference not ready in 10min. Tail:\n{tail(procs['inference'][2], 80)}")
    print("inference is up.")

    start_bg("orchestrator", "python3.12 -m prime_rl.orchestrator.orchestrator @ /kaggle/working/outputs/orch.toml")
    time.sleep(5)
    if procs["orchestrator"][0].poll() is not None:
        raise RuntimeError(f"orchestrator died early. Tail:\n{tail(procs['orchestrator'][2], 80)}")
    print("orchestrator is up.")

    print("\nStarting trainer (live via tee)...")
    print("=" * 60)
    trainer_log = f"{LOG_DIR}/trainer.log"
    t0 = time.time()
    trainer_cmd = (
        "set -o pipefail; "
        "python3.12 -m torch.distributed.run --nproc-per-node 1 "
        "-m prime_rl.trainer.rl.train @ /kaggle/working/outputs/trainer.toml "
        f"2>&1 | tee {trainer_log}"
    )
    trainer = subprocess.run(["bash", "-c", trainer_cmd], env=env)
    elapsed = time.time() - t0
    print("=" * 60)
    print(f"\ntrainer exit={trainer.returncode}  elapsed={elapsed:.1f}s")
    if trainer.returncode != 0:
        print("\n--- inference tail ---");    print(tail(procs["inference"][2], 60))
        print("\n--- orchestrator tail ---"); print(tail(procs["orchestrator"][2], 60))
        raise RuntimeError(f"trainer failed exit={trainer.returncode}")
    outcome = "PASS"
    print("\nGRPO PASS")
finally:
    cleanup()
    print(f"GRPO outcome: {outcome}")


## 4. Eval (base / after_grpo)

GRPO 完了後に GPU が解放されるのを待ってから 2 段階 eval。

In [ ]:
run_eval(MODEL_DIR, label="base")

In [ ]:
import os, shutil, subprocess

GRPO_WEIGHTS_ROOT = "/kaggle/working/outputs/weights"
grpo_steps = sorted(
    [d for d in os.listdir(GRPO_WEIGHTS_ROOT) if d.startswith("step_")],
    key=lambda d: int(d.split("_")[-1]),
) if os.path.isdir(GRPO_WEIGHTS_ROOT) else []

if not grpo_steps:
    print("WARN: no GRPO weights dir found, skipping after_grpo eval")
else:
    GRPO_CKPT_DIR = f"{GRPO_WEIGHTS_ROOT}/{grpo_steps[-1]}"
    print(f"GRPO ckpt: {GRPO_CKPT_DIR}")
    WEIGHT_EXTS = (".safetensors", ".bin", ".pth", ".pt", ".index.json")
    OWNS = {"config.json"}
    copied = []
    for fname in os.listdir(MODEL_DIR):
        src = f"{MODEL_DIR}/{fname}"
        dst = f"{GRPO_CKPT_DIR}/{fname}"
        if not os.path.isfile(src): continue
        if any(fname.endswith(ext) for ext in WEIGHT_EXTS): continue
        if fname in OWNS: continue
        if os.path.exists(dst): continue
        shutil.copy(src, dst)
        copied.append(fname)
    print(f"copied processor files: {copied}")
    run_eval(GRPO_CKPT_DIR, label="after_grpo")

## 5. Summary + cleanup

In [ ]:
import os, shutil, subprocess, json

print("\n" + "=" * 64)
print("EVAL SUMMARY (action_match accuracy on 67 raw-v2 cases)")
print("=" * 64)
print(f"{'stage':<14}{'all':>10}{'positive':>20}{'negative':>20}{'time':>10}")
for r in EVAL_RESULTS:
    print(
        f"{r['label']:<14}"
        f"{r['accuracy']:>9.2%}  "
        f"{r['positive']['acc']:>9.2%} ({r['positive']['correct']}/{r['positive']['n']}) "
        f"{r['negative']['acc']:>9.2%} ({r['negative']['correct']}/{r['negative']['n']}) "
        f"{r['elapsed_s']:>8.1f}s"
    )
with open("/kaggle/working/outputs/eval_results.json", "w") as f:
    json.dump(EVAL_RESULTS, f, indent=2, ensure_ascii=False)

manifest = "/kaggle/working/outputs/manifest.txt"
with open(manifest, "w") as f:
    f.write("# files under /kaggle/working/ at end of notebook\n")
    for root, dirs, files in os.walk("/kaggle/working"):
        for fn in files:
            p = os.path.join(root, fn)
            try: sz = os.path.getsize(p)
            except OSError: sz = -1
            f.write(f"{sz:>12d}  {p}\n")
print("\n=== manifest tail ===")
subprocess.run(f"tail -n 80 {manifest}", shell=True)

for victim in [
    "/kaggle/working/outputs/checkpoints",
    "/kaggle/working/outputs/weights",
    "/kaggle/working/outputs/run_default/broadcasts",
]:
    if os.path.isdir(victim):
        shutil.rmtree(victim)
        print(f"removed: {victim}")

In [ ]:
import os, json, collections

TRACE_PATH = "/kaggle/working/outputs/satelliteagent_traces.jsonl"
print(f"trace file: {TRACE_PATH}")
if not os.path.isfile(TRACE_PATH):
    print("WARN: trace file not found -- env tracer not invoked or env_prep stale")
else:
    sz = os.path.getsize(TRACE_PATH)
    print(f"size: {sz/1024:.1f} KiB")

    rollouts = []
    bad = 0
    with open(TRACE_PATH, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                rollouts.append(json.loads(line))
            except Exception:
                bad += 1
    print(f"rollouts parsed: {len(rollouts)}  (bad lines: {bad})")

    if rollouts:
        terminal_actions = collections.Counter(r.get("terminal_action") for r in rollouts)
        print()
        print("=== terminal_action distribution (per rollout) ===")
        for k, v in terminal_actions.most_common():
            print(f"  {str(k):<20s} {v:>6d}  ({v/len(rollouts):.1%})")

        n_unreached = sum(1 for r in rollouts if not r.get("terminal_called"))
        print()
        print(f"rollouts that did NOT reach a terminal action: {n_unreached} ({n_unreached/len(rollouts):.1%})")

        tool_names = collections.Counter()
        n_calls_per_rollout = []
        for r in rollouts:
            log = r.get("tool_call_log") or []
            n_calls_per_rollout.append(len(log))
            for entry in log:
                tool_names[entry.get("name")] += 1
        print()
        print("=== tool_call_log name distribution (across all rollouts) ===")
        for k, v in tool_names.most_common():
            print(f"  {str(k):<24s} {v:>6d}")
        if n_calls_per_rollout:
            avg = sum(n_calls_per_rollout) / len(n_calls_per_rollout)
            mx = max(n_calls_per_rollout)
            print()
            print(f"turns / rollout: mean={avg:.2f}  max={mx}  (max_turns config=5)")

        n_messages = [r.get("n_messages", 0) for r in rollouts]
        if n_messages:
            print(f"messages / rollout: mean={sum(n_messages)/len(n_messages):.2f}  max={max(n_messages)}")

        n_correct = sum(1 for r in rollouts if r.get("terminal_action") == r.get("expected_action"))
        print()
        print(f"trace-based accuracy (terminal_action == expected_action): {n_correct}/{len(rollouts)} = {n_correct/len(rollouts):.1%}")

        per_case = collections.defaultdict(list)
        for r in rollouts:
            per_case[r.get("case_id")].append(r.get("terminal_action"))
        print()
        print(f"unique case_ids: {len(per_case)}")
        rollouts_per_case = collections.Counter(len(v) for v in per_case.values())
        print(f"rollouts/case histogram: {dict(rollouts_per_case)}")

        fails = [r for r in rollouts if r.get("terminal_action") != r.get("expected_action")]
        print()
        print("=== 3 sample MISMATCH rollouts ===")
        for r in fails[:3]:
            print(f"  case={r.get('case_id')}  expected={r.get('expected_action')}  got={r.get('terminal_action')}  turns={len(r.get('tool_call_log') or [])}")
            for e in (r.get("tool_call_log") or [])[:6]:
                print(f"    -> {e.get('name')}({e.get('args')})")
